In [1]:
# Install transformers, sentencepiece, and datasets libraries
!pip install transformers
!pip install sentencepiece
!pip install datasets


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 480.6/480.6 kB 14.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.3/179.3 kB 18.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/134.8 kB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.1/194.1 kB 18.2 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2024.10.0
    Uninstalling fsspec-2024.10.0:
      Successfully uninstalled fsspec-2024.10.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2024.10.0 requires fsspec==2024.10.0, but you have fsspec 2024.9.0 which is incompatible.


In [2]:
# Install the specific version of fsspec required by gcsfs
!pip install fsspec==2024.10.0


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.6/179.6 kB 5.5 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2024.9.0
    Uninstalling fsspec-2024.9.0:
      Successfully uninstalled fsspec-2024.9.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
datasets 3.1.0 requires fsspec[http]<=2024.9.0,>=2023.1.0, but you have fsspec 2024.10.0 which is incompatible.


In [3]:
# Uninstall the current gcsfs version and install a compatible one
!pip uninstall -y gcsfs
!pip install gcsfs==2023.6.0  # Choose a compatible version that works with fsspec 2024.9.0


Found existing installation: gcsfs 2024.10.0
Uninstalling gcsfs-2024.10.0:
  Successfully uninstalled gcsfs-2024.10.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.8/163.8 kB 7.3 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2024.10.0
    Uninstalling fsspec-2024.10.0:
      Successfully uninstalled fsspec-2024.10.0


In [4]:
# Update datasets to the latest version
!pip install -U datasets


In [5]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [9]:
!pip install evaluate
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, Trainer, TrainingArguments
from datasets import load_dataset, Dataset
import evaluate


# Define dataset paths
train_src = '/content/drive/MyDrive/Code_transfer/train.src'
train_tgt = '/content/drive/MyDrive/Code_transfer/train.tgt'
valid_src = '/content/drive/MyDrive/Code_transfer/valid.src'
valid_tgt = '/content/drive/MyDrive/Code_transfer/valid.tgt'
test_src = '/content/drive/MyDrive/Code_transfer/test.src'
test_tgt = '/content/drive/MyDrive/Code_transfer/test.tgt'


# Load and Prepare Dataset

In [10]:
def load_code_translation_data(src_path, tgt_path):
    with open(src_path, 'r') as src_file, open(tgt_path, 'r') as tgt_file:
        src_lines = src_file.readlines()
        tgt_lines = tgt_file.readlines()
    return Dataset.from_dict({"src": src_lines, "tgt": tgt_lines})

# Load train, validation, and test datasets
train_dataset = load_code_translation_data(train_src, train_tgt)
valid_dataset = load_code_translation_data(valid_src, valid_tgt)
test_dataset = load_code_translation_data(test_src, test_tgt)


# Define Tokenizers and Encoding Techniques

In [12]:
# Initialize the BPE tokenizer
bpe_tokenizer = AutoTokenizer.from_pretrained("Salesforce/codet5-small")

# Define encoding function for BPE
def encode_bpe(example):
    src_enc = bpe_tokenizer(example['src'], padding="max_length", truncation=True, max_length=512)
    tgt_enc = bpe_tokenizer(example['tgt'], padding="max_length", truncation=True, max_length=512)
    return {'input_ids': src_enc['input_ids'], 'attention_mask': src_enc['attention_mask'], 'labels': tgt_enc['input_ids']}

train_bpe = train_dataset.map(encode_bpe)
valid_bpe = valid_dataset.map(encode_bpe)
test_bpe = test_dataset.map(encode_bpe)


Map:   0%|          | 0/1100 [00:00<?, ? examples/s]

Map:   0%|          | 0/275 [00:00<?, ? examples/s]

Map:   0%|          | 0/5 [00:00<?, ? examples/s]

## Positional Encoding

In [13]:
def add_positional_encoding(example):
    src_tokens = bpe_tokenizer(example['src'], padding="max_length", truncation=True, max_length=512)
    tgt_tokens = bpe_tokenizer(example['tgt'], padding="max_length", truncation=True, max_length=512)

    # Manual positional encoding (for demonstration)
    src_tokens['position_ids'] = list(range(128))
    tgt_tokens['position_ids'] = list(range(128))

    return {'input_ids': src_tokens['input_ids'], 'attention_mask': src_tokens['attention_mask'], 'labels': tgt_tokens['input_ids']}

train_positional = train_dataset.map(add_positional_encoding)
valid_positional = valid_dataset.map(add_positional_encoding)
test_positional = test_dataset.map(add_positional_encoding)


Map:   0%|          | 0/1100 [00:00<?, ? examples/s]

Map:   0%|          | 0/275 [00:00<?, ? examples/s]

Map:   0%|          | 0/5 [00:00<?, ? examples/s]

# Word-Wise Encoding

In [14]:
def encode_wordwise(example):
    src_tokens = example['src'].split()
    tgt_tokens = example['tgt'].split()

    # Convert back to single string for tokenization compatibility
    src_enc = bpe_tokenizer(" ".join(src_tokens), padding="max_length", truncation=True, max_length=128)
    tgt_enc = bpe_tokenizer(" ".join(tgt_tokens), padding="max_length", truncation=True, max_length=128)

    return {'input_ids': src_enc['input_ids'], 'attention_mask': src_enc['attention_mask'], 'labels': tgt_enc['input_ids']}

train_wordwise = train_dataset.map(encode_wordwise)
valid_wordwise = valid_dataset.map(encode_wordwise)
test_wordwise = test_dataset.map(encode_wordwise)


Map:   0%|          | 0/1100 [00:00<?, ? examples/s]

Map:   0%|          | 0/275 [00:00<?, ? examples/s]

Map:   0%|          | 0/5 [00:00<?, ? examples/s]

# Define Model and Training Loop

In [20]:
# Load CodeT5 model
model = AutoModelForSeq2SeqLM.from_pretrained("Salesforce/codet5-small")

# Define training arguments
training_args = TrainingArguments(
    output_dir="./results",
    evaluation_strategy="epoch",
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=3,
    logging_dir="./logs",
    logging_steps=100,
    save_steps=10_000,
    save_total_limit=2,
    report_to="none"


)

# Define Trainer for BPE encoding
trainer_bpe = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_bpe,
    eval_dataset=valid_bpe
)

# Train model with BPE encoding
trainer_bpe.train()

## Train model with positional encoding
trainer_positional = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_positional,
    eval_dataset=valid_positional
)
trainer_positional.train()

# Train model with word-wise encoding
trainer_wordwise = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_wordwise,
    eval_dataset=valid_wordwise
)
trainer_wordwise.train()


/usr/local/lib/python3.10/dist-packages/transformers/training_args.py:1525: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


Epoch,Training Loss,Validation Loss
1,0.325400,0.241555
2,0.226300,0.208001
3,0.204500,0.197392


Epoch,Training Loss,Validation Loss
1,0.195700,0.180968
2,0.161600,0.168328
3,0.158200,0.163210


Epoch,Training Loss,Validation Loss
1,0.502700,0.447819
2,0.434600,0.421845
3,0.397200,0.412154


TrainOutput(global_step=825, training_loss=0.4363044068307588, metrics={'train_runtime': 90.5019, 'train_samples_per_second': 36.463, 'train_steps_per_second': 9.116, 'total_flos': 111656986214400.0, 'train_loss': 0.4363044068307588, 'epoch': 3.0})

# Set Up Testing Dataset and Generate Predictions

In [22]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch
from evaluate import load
import matplotlib.pyplot as plt
import nltk
nltk.download('punkt')


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


True

In [24]:
# Save model and tokenizer to ensure both are in the same directory
model.save_pretrained("/content/drive/MyDrive/Code_transfer/codeT5")
tokenizer = AutoTokenizer.from_pretrained("Salesforce/codet5-small")
tokenizer.save_pretrained("/content/drive/MyDrive/Code_transfer/codeT5")


/usr/local/lib/python3.10/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


('/content/drive/MyDrive/Code_transfer/codeT5/tokenizer_config.json',
 '/content/drive/MyDrive/Code_transfer/codeT5/special_tokens_map.json',
 '/content/drive/MyDrive/Code_transfer/codeT5/vocab.json',
 '/content/drive/MyDrive/Code_transfer/codeT5/merges.txt',
 '/content/drive/MyDrive/Code_transfer/codeT5/added_tokens.json',
 '/content/drive/MyDrive/Code_transfer/codeT5/tokenizer.json')

In [25]:
# Define model directory path
model_directory = "/content/drive/MyDrive/Code_transfer/codeT5"

# Load trained model and tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_directory)
model = AutoModelForSeq2SeqLM.from_pretrained(model_directory)


In [26]:
# Test data paths
test_src = '/content/drive/MyDrive/Code_transfer/test.src'
test_tgt = '/content/drive/MyDrive/Code_transfer/test.tgt'

# Load Test Data

In [27]:
def load_test_data(src_path, tgt_path):
    with open(src_path, 'r') as src_file, open(tgt_path, 'r') as tgt_file:
        src_lines = src_file.readlines()
        tgt_lines = tgt_file.readlines()
    return [{"src": src.strip(), "tgt": tgt.strip()} for src, tgt in zip(src_lines, tgt_lines)]

# Load test data
test_data = load_test_data(test_src, test_tgt)


# Generate Predictions for Each Encoding Scheme

In [35]:
def generate_predictions_and_references(test_data, encoding_func):
    predictions = []
    references = []

    for example in test_data:
        # Encode using the specified encoding function
        encoded_input = encoding_func({"src": example["src"], "tgt": example["tgt"]})
        input_ids = torch.tensor(encoded_input["input_ids"]).unsqueeze(0).to(model.device)
        attention_mask = torch.tensor(encoded_input["attention_mask"]).unsqueeze(0).to(model.device)

        # Generate prediction
        with torch.no_grad():
            output = model.generate(input_ids=input_ids, attention_mask=attention_mask)

        # Decode and store
        decoded_pred = tokenizer.decode(output[0], skip_special_tokens=True)
        decoded_ref = example["tgt"]
        predictions.append(decoded_pred)
        references.append(decoded_ref)

    # Print final predictions and references for debugging
    print(f"Predictions: {predictions}")
    print(f"References: {references}")

    return predictions, references


In [31]:
# Define encoding functions for BPE, Positional, and Word-Wise
def encode_bpe(example):
    src_enc = tokenizer(example['src'], padding="max_length", truncation=True, max_length=512)
    return {'input_ids': src_enc['input_ids'], 'attention_mask': src_enc['attention_mask']}

def add_positional_encoding(example):
    src_enc = tokenizer(example['src'], padding="max_length", truncation=True, max_length=512)
    src_enc['position_ids'] = list(range(128))
    return {'input_ids': src_enc['input_ids'], 'attention_mask': src_enc['attention_mask']}

def encode_wordwise(example):
    src_tokens = example['src'].split()
    src_enc = tokenizer(" ".join(src_tokens), padding="max_length", truncation=True, max_length=512)
    return {'input_ids': src_enc['input_ids'], 'attention_mask': src_enc['attention_mask']}


In [38]:
def generate_predictions_and_references(test_data, encoding_function, max_new_tokens=128):
    predictions = []
    references = []

    # Generate predictions
    for example in test_data:
        # Apply encoding function to the example
        encoded_example = encoding_function(example)

        # Print the encoded input for debugging
        print(f"Encoded Example: {encoded_example}")

        # Convert to tensors and move to the model's device
        input_ids = torch.tensor(encoded_example['input_ids']).unsqueeze(0).to(model.device)
        attention_mask = torch.tensor(encoded_example['attention_mask']).unsqueeze(0).to(model.device)

        # Generate model output with max_new_tokens to control output length
        with torch.no_grad():
            outputs = model.generate(input_ids=input_ids, attention_mask=attention_mask, max_new_tokens=max_new_tokens)

        # Print the output for debugging
        print(f"Generated Output: {outputs}")

        # Decode predictions
        decoded_preds = tokenizer.decode(outputs[0], skip_special_tokens=True)

        # Decode references (this should be the original target text from the test set)
        decoded_refs = example['tgt']  # Use the original target text as references

        predictions.append(decoded_preds)
        references.append(decoded_refs)

    # Print final predictions and references for debugging
    print(f"Predictions: {predictions}")
    print(f"References: {references}")

    return predictions, references

# Generate predictions and references for each encoding scheme
predictions_bpe, references_bpe = generate_predictions_and_references(test_data, encode_bpe, max_new_tokens=512)
predictions_positional, references_positional = generate_predictions_and_references(test_data, add_positional_encoding, max_new_tokens=512)
predictions_wordwise, references_wordwise = generate_predictions_and_references(test_data, encode_wordwise, max_new_tokens=512)


Encoded Example: {'input_ids': [1, 2978, 1241, 67, 1153, 261, 269, 513, 67, 1937, 16, 269, 513, 67, 8734, 16, 269, 513, 67, 20390, 16, 269, 513, 67, 1672, 16, 269, 8632, 16, 269, 813, 67, 86, 16, 269, 1022, 337, 30752, 21, 16, 269, 1022, 337, 30752, 21, 67, 438, 67, 90, 1236, 16, 269, 513, 67, 2463, 269, 11272, 269, 810, 513, 67, 1937, 31, 269, 876, 513, 67, 8734, 31, 269, 876, 513, 67, 20390, 31, 269, 876, 513, 67, 1672, 31, 269, 810, 306, 6938, 30, 20, 65, 8632, 31, 269, 810, 306, 6938, 30, 20, 65, 813, 67, 86, 31, 269, 876, 306, 6938, 30, 20, 65, 1022, 337, 30752, 21, 31, 269, 876, 1022, 337, 30752, 21, 67, 438, 67, 90, 1236, 31, 269, 876, 306, 6938, 30, 20, 65, 513, 67, 2463, 31, 269, 960, 1022, 337, 30752, 21, 67, 438, 67, 90, 1236, 31, 269, 6636, 306, 6938, 30, 20, 65, 3888, 67, 27074, 67, 7950, 67, 84, 22, 31, 269, 6636, 513, 67, 311, 67, 1574, 31, 269, 1241, 67, 1153, 67, 16411, 67, 1578, 87, 67, 1578, 87, 67, 1578, 67, 21, 67, 21, 468, 12, 269, 263, 734, 12, 404, 262, 16, 269,

In [42]:
# Define encoding functions for BPE, Positional, and Word-Wise
def encode_bpe(example):
    src_enc = tokenizer(example['src'], padding="max_length", truncation=True, max_length=512)
    return {'input_ids': src_enc['input_ids'], 'attention_mask': src_enc['attention_mask']}

def add_positional_encoding(example):
    src_enc = tokenizer(example['src'], padding="max_length", truncation=True, max_length=512)
    src_enc['position_ids'] = list(range(512))
    return {'input_ids': src_enc['input_ids'], 'attention_mask': src_enc['attention_mask']}

def encode_wordwise(example):
    src_tokens = example['src'].split()
    src_enc = tokenizer(" ".join(src_tokens), padding="max_length", truncation=True, max_length=512)
    return {'input_ids': src_enc['input_ids'], 'attention_mask': src_enc['attention_mask']}


In [43]:
# Generate predictions and references for each encoding scheme
predictions_bpe, references_bpe = generate_predictions_and_references(test_data, encode_bpe, max_new_tokens=512)
predictions_positional, references_positional = generate_predictions_and_references(test_data, add_positional_encoding, max_new_tokens=512)
predictions_wordwise, references_wordwise = generate_predictions_and_references(test_data, encode_wordwise, max_new_tokens=512)


Encoded Example: {'input_ids': [1, 2978, 1241, 67, 1153, 261, 269, 513, 67, 1937, 16, 269, 513, 67, 8734, 16, 269, 513, 67, 20390, 16, 269, 513, 67, 1672, 16, 269, 8632, 16, 269, 813, 67, 86, 16, 269, 1022, 337, 30752, 21, 16, 269, 1022, 337, 30752, 21, 67, 438, 67, 90, 1236, 16, 269, 513, 67, 2463, 269, 11272, 269, 810, 513, 67, 1937, 31, 269, 876, 513, 67, 8734, 31, 269, 876, 513, 67, 20390, 31, 269, 876, 513, 67, 1672, 31, 269, 810, 306, 6938, 30, 20, 65, 8632, 31, 269, 810, 306, 6938, 30, 20, 65, 813, 67, 86, 31, 269, 876, 306, 6938, 30, 20, 65, 1022, 337, 30752, 21, 31, 269, 876, 1022, 337, 30752, 21, 67, 438, 67, 90, 1236, 31, 269, 876, 306, 6938, 30, 20, 65, 513, 67, 2463, 31, 269, 960, 1022, 337, 30752, 21, 67, 438, 67, 90, 1236, 31, 269, 6636, 306, 6938, 30, 20, 65, 3888, 67, 27074, 67, 7950, 67, 84, 22, 31, 269, 6636, 513, 67, 311, 67, 1574, 31, 269, 1241, 67, 1153, 67, 16411, 67, 1578, 87, 67, 1578, 87, 67, 1578, 67, 21, 67, 21, 468, 12, 269, 263, 734, 12, 404, 262, 16, 269,

In [45]:
# Install the required jiwer library for WER computation
!pip install jiwer


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 23.7 MB/s eta 0:00:00


In [46]:
# Load WER metric
##wer_metric = load("wer")

# Compute WER for each encoding scheme
##wer_bpe = wer_metric.compute(predictions=predictions_bpe, references=references_bpe)
##wer_positional = wer_metric.compute(predictions=predictions_positional, references=references_positional)
##wer_wordwise = wer_metric.compute(predictions=predictions_wordwise, references=references_wordwise)

# Print WER scores
##print(f"WER for BPE encoding: {wer_bpe}")
##print(f"WER for Positional encoding: {wer_positional}")
##print(f"WER for Word-wise encoding: {wer_wordwise}")


WER for BPE encoding: 0.4451345755693582
WER for Positional encoding: 0.4451345755693582
WER for Word-wise encoding: 0.4451345755693582


In [51]:
!pip install nltk
import nltk
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction


In [52]:
def calculate_bleu_score(predictions, references):
    # Use a smoothing function to avoid zero scores on shorter texts
    smoothing_function = SmoothingFunction().method1
    bleu_scores = []

    for pred, ref in zip(predictions, references):
        # Tokenize predictions and references
        pred_tokens = pred.split()
        ref_tokens = [ref.split()]  # reference must be a list of lists
        score = sentence_bleu(ref_tokens, pred_tokens, smoothing_function=smoothing_function)
        bleu_scores.append(score)

    # Calculate average BLEU score across all predictions
    avg_bleu_score = sum(bleu_scores) / len(bleu_scores)
    return avg_bleu_score


In [53]:
# Calculate BLEU score for BPE encoding
bleu_bpe = calculate_bleu_score(predictions_bpe, references_bpe)
print(f"BLEU score for BPE encoding: {bleu_bpe}")

# Calculate BLEU score for Positional encoding
bleu_positional = calculate_bleu_score(predictions_positional, references_positional)
print(f"BLEU score for Positional encoding: {bleu_positional}")

# Calculate BLEU score for Word-wise encoding
bleu_wordwise = calculate_bleu_score(predictions_wordwise, references_wordwise)
print(f"BLEU score for Word-wise encoding: {bleu_wordwise}")


BLEU score for BPE encoding: 0.4539000545119135
BLEU score for Positional encoding: 0.4539000545119135
BLEU score for Word-wise encoding: 0.4539000545119135
